In [1]:
import pandas as pd

doc = pd.read_excel("manual15000business_change_statements.xlsx")

print(f"Number of entities: {doc['QID'].nunique()}")
print(f"Number of entity-property pairs: "
      f"{doc.groupby(['QID', 'Property_ID']).ngroups}")

Number of entities: 7507
Number of entity-property pairs: 12940


In [3]:
change_qualifier_ids = {'P12413', 'P12506', 'P1264', 'P1319', 'P1326', 'P1365', 'P1366', 'P156', 'P2754', 'P3415',
                        'P571', 'P576', 'P577', 'P580', 'P582', 'P585', 'P7888', 'P8554', 'P8555'}

qual_id_cols = [c for c in doc.columns if c.startswith('Qualifier_Property_ID')]

def has_change_qual(row):
    for col in qual_id_cols:
        v = row[col]
        if pd.notna(v) and str(v).strip() in change_qualifier_ids:
            return True
    return False

doc_class1 = doc[doc['change_class'] == 'class1'].copy()
doc_class2 = doc[doc['change_class'] == 'class2'].copy()

# Approach: (property+value) + qualifier + rank + reference

# 1: label each statement
doc_class1['has_cq']  = doc_class1.apply(has_change_qual, axis=1)
doc_class1['has_rk']  = doc_class1['Rank'].isin(['preferred', 'deprecated'])
doc_class1['has_ref'] = doc_class1['Has_References'] == 'Yes'

doc_class2['has_cq']  = doc_class2.apply(has_change_qual, axis=1)
doc_class2['has_rk']  = doc_class2['Rank'].isin(['preferred', 'deprecated'])
doc_class2['has_ref'] = doc_class2['Has_References'] == 'Yes'

# 2: label approach for each pair
def make_approach(row):
    parts = ['Property + Value']
    if row['Q']:   parts.append('Qualifier')
    if row['Rk']:  parts.append('Rank')
    if row['Ref']: parts.append('Reference')
    return ' + '.join(parts)

def get_pair_approach(data, label):
    pair_stats = data.groupby(['QID', 'Property_ID']).agg(
        Q   = ('has_cq',  'any'),
        Rk  = ('has_rk',  'any'),
        Ref = ('has_ref', 'any'),
    ).reset_index()
    pair_stats['approach'] = pair_stats.apply(make_approach, axis=1)
    pair_stats['label']    = label
    return pair_stats

pair_c1 = get_pair_approach(doc_class1, 'class1')
pair_c2 = get_pair_approach(doc_class2, 'class2')

def count_approach(pair_stats):
    total  = len(pair_stats)
    counts = pair_stats['approach'].value_counts().reset_index()
    counts.columns = ['approach', 'count']
    counts['percentage'] = (counts['count'] / total * 100).round(2).astype(str) + '%'
    return counts

result_c1  = count_approach(pair_c1)
result_c2  = count_approach(pair_c2)
pair_all   = pd.concat([pair_c1, pair_c2], ignore_index=True)
result_all = count_approach(pair_all)


output_path = '0618business_approach_analysis.xlsx'
with pd.ExcelWriter(output_path) as writer:
    result_c1.to_excel(writer,  sheet_name='class1_approach',  index=False)
    result_c2.to_excel(writer,  sheet_name='class2_approach',  index=False)
    result_all.to_excel(writer, sheet_name='overall_approach', index=False)
    pair_all.to_excel(writer,   sheet_name='pair_details',     index=False)

print(f"saved: 0618business_approach_analysis.xlsx")
print(f"\nclass1 pairs: {len(pair_c1)}")
print(f"class2 pairs: {len(pair_c2)}")
print(f"total pairs:  {len(pair_all)}")

saved: 0618business_approach_analysis.xlsx

class1 pairs: 7342
class2 pairs: 5598
total pairs:  12940
